In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta
import pickle as pkl

from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data" / "speeches.parquet").is_file()
)
FIG_DIR = PROJECT_ROOT / "results" / "figures"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

input_path = PROJECT_ROOT / "data" / "data_with_topics.pkl"
if input_path.exists():
    df = pd.read_pickle(input_path)
else:
    df = pd.read_parquet(PROJECT_ROOT / "data" / "speeches.parquet")

topic_path = PROJECT_ROOT / "data" / "policy_topics.parquet"
if "topic_label" not in df.columns and topic_path.exists():
    topics = pd.read_parquet(topic_path)
    df = df.merge(topics[["original_row_idx", "topic_label"]],
                  on="original_row_idx", how="left", validate="one_to_one")

df['moral_rating'] = pd.to_numeric(df['moral_rating'], errors='coerce')
df = df.dropna(subset=['moral_rating'])
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['year'] = df['date'].dt.year
PARTY_SV_NORMALIZE = {
    "Moderata samlingspartiet": "Moderaterna",
    "Folkpartiet liberalerna": "Liberalerna",
    "Folkpartiet": "Liberalerna",
    "Kristdemokratiska Samhällspartiet": "Kristdemokraterna",
    "Vänsterpartiet Kommunisterna": "Vänsterpartiet",
}

REMOVE_PARTIES = {"Utan partibeteckning", "no party"}
df.head()

In [ ]:
# only include from 1991 election onward
df = df[df['moral_rating'] != 0]
df =df[df['date'] >= pd.to_datetime("1991-09-15")]
df['party_name_norm'] = df['party_name'].replace(PARTY_SV_NORMALIZE)
df_full = df.copy()
df = df[~df['party_name_norm'].isin(REMOVE_PARTIES)]
print(len(df))

In [ ]:
print(df.columns)

In [ ]:
import pandas as pd
import numpy as np


# Ensure dates are datetime
df_full["date"] = pd.to_datetime(df_full["date"], errors="coerce")
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Create year if necessary
df_full["year"] = df_full["date"].dt.year
df["year"] = df["date"].dt.year


if "word_count" not in df.columns:
    df["word_count"] = df["text"].fillna("").astype(str).str.split().str.len()


# Formatting functions
def format_integer(x):
    """Format counts with thousands separators."""
    if pd.isna(x):
        return ""
    return f"{int(round(x)):,}"


def format_decimal(x, decimals=1):
    """Format decimal values."""
    if pd.isna(x):
        return ""
    return f"{x:,.{decimals}f}"


def escape_latex(text):
    """Escape common LaTeX special characters."""
    replacements = {
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
    }

    text = str(text)
    for old, new in replacements.items():
        text = text.replace(old, new)

    return text

In [ ]:

n_full = len(df_full)
n_missing_party = len(df_full)-len(df)
n_final = len(df)

table_a1 = pd.DataFrame({
    "Sample": [
        "SWERIK speeches, 1991--2023",
        "Speeches excluded due to missing party information",
        "Final analytical corpus"
    ],
    "Speeches": [
        n_full,
        n_missing_party,
        n_final
    ],
    "Percent of original corpus": [
        100,
        100 * n_missing_party / n_full if n_full else np.nan,
        100 * n_final / n_full if n_full else np.nan
    ]
})

table_a1_display = table_a1.copy()
table_a1_display["Speeches"] = table_a1_display["Speeches"].map(format_integer)
table_a1_display["Percent of original corpus"] = (
    table_a1_display["Percent of original corpus"]
    .map(lambda x: format_decimal(x, 1))
)

latex_a1 = table_a1_display.to_latex(
    index=False,
    escape=False,
    column_format="lrr",
    caption="Construction of the analytical corpus.",
    label="tab:corpus_construction"
)

print(latex_a1)

In [ ]:
print(n_missing_party)

In [ ]:
# ============================================================
# TABLE A2: OVERALL CORPUS DESCRIPTION
# ============================================================

# Number of speeches per speaker
speeches_per_speaker = df.groupby("person_id").size()

overall_statistics = [
    ("Speeches", len(df), "integer"),
    ("Unique speakers", df["person_id"].nunique(), "integer"),
    ("Years covered", f"{int(df['year'].min())}--{int(df['year'].max())}", "text"),
    ("Mean speeches per speaker", speeches_per_speaker.mean(), "decimal"),
    ("Median speeches per speaker", speeches_per_speaker.median(), "decimal"),
    (
        "10th--90th percentile, speeches per speaker",
        (
            speeches_per_speaker.quantile(0.10),
            speeches_per_speaker.quantile(0.90)
        ),
        "range"
    ),
    ("Mean speech length, words", df["word_count"].mean(), "decimal"),
    ("Median speech length, words", df["word_count"].median(), "decimal"),
    (
        "10th--90th percentile, speech length",
        (
            df["word_count"].quantile(0.10),
            df["word_count"].quantile(0.90)
        ),
        "range"
    )
]


def format_statistic(value, value_type):
    if value_type == "integer":
        return format_integer(value)

    if value_type == "decimal":
        return format_decimal(value, 1)

    if value_type == "range":
        lower, upper = value
        return f"{format_decimal(lower, 1)}--{format_decimal(upper, 1)}"

    return str(value)


table_a2 = pd.DataFrame({
    "Measure": [row[0] for row in overall_statistics],
    "Value": [
        format_statistic(row[1], row[2])
        for row in overall_statistics
    ]
})

latex_a2 = table_a2.to_latex(
    index=False,
    escape=False,
    column_format="lr",
    caption="Descriptive statistics for the final analytical corpus.",
    label="tab:overall_corpus"
)

rows_a2 = "\n".join(
    f"{escape_latex(measure)} & {format_statistic(value, value_type)} \\\\"
    for measure, value, value_type in overall_statistics
)

latex_a2 = rf"""
\begin{{table}}[!htbp]
\centering
\caption{{Descriptive statistics for the final analytical corpus}}
\label{{tab:overall_corpus}}
\begin{{threeparttable}}
\begin{{tabular}}{{lr}}
\toprule
Measure & Value \\
\midrule
{rows_a2}
\bottomrule
\end{{tabular}}
\begin{{tablenotes}}[flushleft]
\footnotesize
\item \textit{{Note:}} Speech length is measured as the number of
whitespace-separated words. Speaker-level statistics are calculated using
unique \texttt{{person\_id}} values.
\end{{tablenotes}}
\end{{threeparttable}}
\end{{table}}
"""

print(latex_a2)

In [ ]:
# Ensure moral_rating is numeric
df["moral_rating"] = pd.to_numeric(
    df["moral_rating"],
    errors="coerce"
)


# ============================================================
# PANEL A: CORPUS BY YEAR
# ============================================================

year_table = (
    df.groupby("year")
    .agg(
        Speeches=("moral_rating", "size"),
        Unique_speakers=("person_id", "nunique"),
        Mean_moral_rating=("moral_rating", "mean"),
        SD_moral_rating=("moral_rating", "std")
    )
    .reset_index()
    .sort_values("year")
)

year_table["year"] = year_table["year"].astype(int)


# ============================================================
# PANEL B: CORPUS BY PARTY
# ============================================================

party_table = (
    df.groupby("party_name_norm")
    .agg(
        Speeches=("moral_rating", "size"),
        Unique_speakers=("person_id", "nunique"),
        Mean_moral_rating=("moral_rating", "mean"),
        SD_moral_rating=("moral_rating", "std")
    )
    .reset_index()
    .rename(columns={"party_name_norm": "Party"})
)

party_table["Share"] = (
    100 * party_table["Speeches"] / len(df)
)

party_table = (
    party_table
    .sort_values("Speeches", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
# ============================================================
# LATEX ROWS: PANEL A
# ============================================================

year_rows = []

for _, row in year_table.iterrows():
    year_rows.append(
        f"{int(row['year'])} & "
        f"{format_integer(row['Speeches'])} & "
        f"{format_integer(row['Unique_speakers'])} & "
        f"{format_decimal(row['Mean_moral_rating'], 2)} & "
        f"{format_decimal(row['SD_moral_rating'], 2)} \\\\"
    )

year_rows.append(r"\midrule")

year_rows.append(
    f"Total & "
    f"{format_integer(len(df))} & "
    f"{format_integer(df['person_id'].nunique())} & "
    f"{format_decimal(df['moral_rating'].mean(), 2)} & "
    f"{format_decimal(df['moral_rating'].std(), 2)} \\\\"
)

year_rows = "\n".join(year_rows)


# ============================================================
# LATEX ROWS: PANEL B
# ============================================================

party_rows = []

for _, row in party_table.iterrows():
    party_rows.append(
        f"{escape_latex(row['Party'])} & "
        f"{format_integer(row['Speeches'])} & "
        f"{format_integer(row['Unique_speakers'])} & "
        f"{format_decimal(row['Share'], 1)} & "
        f"{format_decimal(row['Mean_moral_rating'], 2)} & "
        f"{format_decimal(row['SD_moral_rating'], 2)} \\\\"
    )

party_rows = "\n".join(party_rows)

latex_a3 = rf"""
\begin{{table}}[!htbp]
\centering
\caption{{Composition of the final analytical corpus by year and party}}
\label{{tab:corpus_year_party}}

\textbf{{Panel A: By year}}

\vspace{{0.3em}}

\begin{{tabular}}{{rrrrr}}
\toprule
Year & Speeches & Unique speakers & Mean rating & SD rating \\
\midrule
{year_rows}
\bottomrule
\end{{tabular}}

\vspace{{1.2em}}

\textbf{{Panel B: By party}}

\vspace{{0.3em}}

\begin{{tabular}}{{lrrrrr}}
\toprule
Party & Speeches & Unique speakers & Share (\%) & Mean rating & SD rating \\
\midrule
{party_rows}
\bottomrule
\end{{tabular}}

\vspace{{0.4em}}

\begin{{minipage}}{{0.95\textwidth}}
\footnotesize
\textit{{Note:}} The table describes the final analytical corpus after
excluding speeches for which party information is unavailable. Unique speakers
are identified using \texttt{{person\_id}}. Mean and median moral ratings are
calculated using non-missing values of \texttt{{moral\_rating}}.
\end{{minipage}}

\end{{table}}
"""

print(latex_a3)

In [ ]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

oldest_date = df["date"].min()
newest_date = df["date"].max()

print("Oldest date:", oldest_date)
print("Newest date:", newest_date)

In [ ]:
table_a1.to_csv(TABLE_DIR / "table_a1_1.csv", index=False)
table_a2.to_csv(TABLE_DIR / "table_a1_2.csv", index=False)
year_table.to_csv(TABLE_DIR / "table_a1_3_year.csv", index=False)
party_table.to_csv(TABLE_DIR / "table_a1_3_party.csv", index=False)
